# 02. Build chronological global Elo features

Read notebook 01's `matches_clean.csv` and produce pre-match global ratings and win probabilities for every accepted game. Exact team names identify teams; new teams start at 1500 and ratings carry across years without a season reset.

The timing rule is **predict first, update afterward**. Games at equal timestamps share frozen state, then their changes are summed. Outputs are `elo_features_all_years.csv` and a final rating summary. Earlier evaluation results can update later ratings; this is not a forecast frozen before a tournament.


## 01. Locate the clean history

Import numerical tools and require the clean match export from notebook 01. Read and write within the processed-data folder. A missing input stops execution with a useful error.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

base = Path.cwd()

if not (base / "data" / "processed").is_dir():
    base = base.parent

input_path = base / "data" / "processed" / "matches_clean.csv"
output_dir = base / "data" / "processed"

if not input_path.is_file():
    raise FileNotFoundError(
        f"Missing {input_path}. Run notebook 01 first."
    )

print("Input:", input_path)

Input: C:\Users\USER\lol_worlds_predictor\data\processed\matches_clean.csv


## 02. Read required match fields

Load the six history columns, parse timestamps as UTC, and convert A's result to numeric form. Read team and match identifiers as strings. Inspect the preview before constructing ratings.


In [2]:
matches = pd.read_csv(
    input_path,
    dtype={
        "match_id": "string",
        "league": "string",
        "team_a": "string",
        "team_b": "string",
    },
)

required_columns = [
    "match_id",
    "date",
    "league",
    "team_a",
    "team_b",
    "result_a",
]

missing_columns = set(required_columns) - set(matches.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

matches = matches[required_columns].copy()

matches["date"] = pd.to_datetime(
    matches["date"],
    utc=True,
    errors="raise",
    format="mixed",
)

matches["result_a"] = pd.to_numeric(
    matches["result_a"],
    errors="raise",
)

display(matches.head())

,match_id,date,league,team_a,team_b,result_a
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0


## 03. Validate and order the input

Reject empty data, missing or blank fields, duplicate IDs, nonbinary outcomes, and self-matches. Sort by date and match ID for reproducibility. These checks protect the state-update loop from malformed records.


In [3]:
if matches.empty:
    raise ValueError("The match table is empty.")

if matches.isna().any().any():
    raise ValueError("Required fields contain missing values.")

for column in ["match_id", "league", "team_a", "team_b"]:
    matches[column] = matches[column].str.strip()

    if matches[column].eq("").any():
        raise ValueError(f"Empty values found in {column}.")

if not matches["match_id"].is_unique:
    raise ValueError("Duplicate match IDs found.")

if not matches["result_a"].isin([0, 1]).all():
    raise ValueError("Results must be 0 or 1.")

if matches["team_a"].eq(matches["team_b"]).any():
    raise ValueError("A team cannot play against itself.")

matches["result_a"] = matches["result_a"].astype("int64")

matches = (
    matches.sort_values(["date", "match_id"])
    .reset_index(drop=True)
)

print("Matches:", len(matches))
print("From:", matches["date"].min())
print("Through:", matches["date"].max())

display(
    matches.groupby(matches["date"].dt.year)
    .size()
    .rename("games")
)

Matches: 96075
From: 2017-01-10 15:44:06+00:00
Through: 2026-10-26 21:54:37+00:00


date
2017     5504
2018     6734
2019     8125
2020     9740
2021    12291
2022    12519
2023    11097
2024    10203
2025    10044
2026     9818
Name: games, dtype: int64

## 04. Define Elo probabilities and transfers

Use initial Elo 1500, K=24, and scale=400. The win probability is `1 / (1 + 10 ** ((rating_b - rating_a) / 400))`; A gains `24 * (result_a - probability_a)` and B loses the same amount. Each game therefore conserves total rating.


In [4]:
INITIAL_ELO = 1500.0
K_FACTOR = 24.0
ELO_SCALE = 400.0

def expected_score(rating_a, rating_b):
    """Calculate A's expected probability of winning."""
    return 1.0 / (
        1.0 + 10.0 ** ((rating_b - rating_a) / ELO_SCALE)
    )

def update_elo(rating_a, rating_b, result_a, k=K_FACTOR):
    """Return updated ratings for A and B."""
    if result_a not in (0, 1):
        raise ValueError("result_a must be 0 or 1.")

    probability_a = expected_score(rating_a, rating_b)
    delta = k * (result_a - probability_a)

    return rating_a + delta, rating_b - delta

## 05. Check the Elo arithmetic

Verify probability 0.5 for equal ratings, a higher probability for the stronger opponent, and complementary probabilities when teams swap. Equal-rated wins and losses transfer 12 points. These are mathematical checks rather than evaluation results.


In [5]:
assert np.isclose(expected_score(1500, 1500), 0.5)
assert expected_score(1600, 1500) > 0.5

assert np.isclose(
    expected_score(1600, 1500)
    + expected_score(1500, 1600),
    1.0,
)

np.testing.assert_allclose(
    update_elo(1500, 1500, result_a=1),
    [1512, 1488],
)

np.testing.assert_allclose(
    update_elo(1500, 1500, result_a=0),
    [1488, 1512],
)

updated_a, updated_b = update_elo(1700, 1450, result_a=0)

assert np.isclose(updated_a + updated_b, 1700 + 1450)

print("Elo calculation checks passed.")

Elo calculation checks passed.


## 06. Record prior features and update timestamp batches

Traverse chronological timestamp batches and record prior ratings, their difference, and probability before using results. Accumulate changes inside the batch and apply them afterward. Simultaneous results cannot affect one another's predictions.


In [6]:
def generate_elo_features(
    match_table,
    initial_elo=INITIAL_ELO,
    k=K_FACTOR,
):
    ratings = {}
    feature_records = []

    ordered_matches = match_table.sort_values(
        ["date", "match_id"]
    )

    for timestamp, batch in ordered_matches.groupby("date", sort=True):
        pending_changes = {}

        for row in batch.itertuples(index=False):
            pre_elo_a = ratings.get(row.team_a, initial_elo)
            pre_elo_b = ratings.get(row.team_b, initial_elo)

            probability_a = expected_score(
                pre_elo_a,
                pre_elo_b,
            )

            feature_records.append(
                {
                    "match_id": row.match_id,
                    "date": timestamp,
                    "league": row.league,
                    "team_a": row.team_a,
                    "team_b": row.team_b,
                    "result_a": row.result_a,
                    "pre_elo_a": pre_elo_a,
                    "pre_elo_b": pre_elo_b,
                    "elo_diff": pre_elo_a - pre_elo_b,
                    "elo_probability_a": probability_a,
                }
            )

            new_elo_a, new_elo_b = update_elo(
                pre_elo_a,
                pre_elo_b,
                row.result_a,
                k=k,
            )

            pending_changes[row.team_a] = (
                pending_changes.get(row.team_a, 0.0)
                + new_elo_a
                - pre_elo_a
            )

            pending_changes[row.team_b] = (
                pending_changes.get(row.team_b, 0.0)
                + new_elo_b
                - pre_elo_b
            )

        for team, change in pending_changes.items():
            ratings[team] = ratings.get(team, initial_elo) + change

    return pd.DataFrame(feature_records), ratings

## 07. Test chronology with synthetic games

Construct A/B/C examples to check simultaneous games, carryover across a year boundary, and independence from the predicted game's own result. These fixtures are artificial test input, not match data. Passing assertions verifies the intended timing.


In [7]:
example_matches = pd.DataFrame(
    {
        "match_id": ["example_1", "example_2", "example_3"],
        "date": pd.to_datetime(
            [
                "2023-12-31 12:00:00",
                "2023-12-31 12:00:00",
                "2024-01-01 12:00:00",
            ],
            utc=True,
        ),
        "league": ["Example", "Example", "Example"],
        "team_a": ["A", "A", "A"],
        "team_b": ["B", "C", "B"],
        "result_a": [1, 0, 1],
    }
)

example_features, _ = generate_elo_features(example_matches)

# Both simultaneous games see A's initial rating.
assert np.isclose(example_features.loc[0, "pre_elo_a"], 1500)
assert np.isclose(example_features.loc[1, "pre_elo_a"], 1500)

# A's equal-rating win and loss cancel.
assert np.isclose(example_features.loc[2, "pre_elo_a"], 1500)

# B retains its previous loss across the year boundary.
assert np.isclose(example_features.loc[2, "pre_elo_b"], 1488)

# Changing the final result cannot change its own pre-match features.
changed_example = example_matches.copy()
changed_example.loc[2, "result_a"] = 0

changed_features, _ = generate_elo_features(changed_example)

elo_columns = [
    "pre_elo_a",
    "pre_elo_b",
    "elo_diff",
    "elo_probability_a",
]

np.testing.assert_allclose(
    example_features[elo_columns].to_numpy(),
    changed_features[elo_columns].to_numpy(),
)

print("Chronology and year-boundary checks passed.")

display(example_features)

Chronology and year-boundary checks passed.


,match_id,date,league,team_a,team_b,result_a,pre_elo_a,pre_elo_b,elo_diff,elo_probability_a
0,example_1,2023-12-31 12:00:00+00:00,Example,A,B,1,1500.0,1500.0,0.0,0.500000
1,example_2,2023-12-31 12:00:00+00:00,Example,A,C,0,1500.0,1500.0,0.0,0.500000
2,example_3,2024-01-01 12:00:00+00:00,Example,A,B,1,1500.0,1488.0,12.0,0.517263


## 08. Replay the complete history

Run the feature generator over all accepted games. Each output row retains the actual outcome for evaluation alongside features recorded before that outcome was applied. The returned rating dictionary represents state after the whole history.


In [8]:
features_df, final_ratings = generate_elo_features(matches)

print("Feature rows:", len(features_df))
print("Teams rated:", len(final_ratings))

display(features_df.head(10))

Feature rows: 96075
Teams rated: 2250


,match_id,date,league,team_a,team_b,result_a,pre_elo_a,pre_elo_b,elo_diff,elo_probability_a
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0,1500.0,1500.0,0.0,0.500000
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0,1500.0,1500.0,0.0,0.500000
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0,1500.0,1500.0,0.0,0.500000
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1,1500.0,1500.0,0.0,0.500000
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0,1500.0,1500.0,0.0,0.500000
5,2017-01-11 15:33:10+00:00|EU CS|TREU/1001490139,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,1,1512.0,1512.0,0.0,0.500000
6,2017-01-11 15:46:32+00:00|EU CS|TREU/1001490141,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,0,1500.0,1488.0,12.0,0.517263
7,2017-01-11 16:44:20+00:00|EU CS|TREU/1001490143,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,0,1488.0,1512.0,-24.0,0.465516
8,2017-01-11 17:53:03+00:00|EU CS|TREU/1001490147,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,0,1488.0,1500.0,-12.0,0.482737
9,2017-01-11 18:59:33+00:00|EU CS|TREU/1001490149,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,0,1488.0,1488.0,0.0,0.500000


## 09. Validate the generated features

Check row preservation, unique IDs, chronological order, finite features, valid probabilities, and the rating-difference formula. Verify rating conservation and that each team begins at 1500 at its first timestamp. These checks detect timing and bookkeeping errors.


In [9]:
assert len(features_df) == len(matches)
assert features_df["match_id"].is_unique
assert features_df["date"].is_monotonic_increasing
assert features_df.notna().all().all()

assert np.isfinite(
    features_df[elo_columns].to_numpy()
).all()

assert features_df["elo_probability_a"].between(0, 1).all()

np.testing.assert_allclose(
    features_df["elo_diff"],
    features_df["pre_elo_a"] - features_df["pre_elo_b"],
)

# Every update transfers rating points between opponents.
assert np.isclose(
    sum(final_ratings.values()),
    INITIAL_ELO * len(final_ratings),
)

# Every team's first timestamp must start at the initial rating.
appearances = pd.concat(
    [
        features_df[["date", "team_a", "pre_elo_a"]].rename(
            columns={"team_a": "team", "pre_elo_a": "pre_elo"}
        ),
        features_df[["date", "team_b", "pre_elo_b"]].rename(
            columns={"team_b": "team", "pre_elo_b": "pre_elo"}
        ),
    ],
    ignore_index=True,
)

first_timestamp = appearances.groupby("team")["date"].transform("min")

first_appearances = appearances.loc[
    appearances["date"].eq(first_timestamp)
]

np.testing.assert_allclose(
    first_appearances["pre_elo"],
    INITIAL_ELO,
)

print("Full-history feature checks passed.")

Full-history feature checks passed.


## 10. Summarize final ratings

Build a final rating table and attach each team's last match date. Inactive teams remain because this baseline has no inactivity decay. Never join end-of-history ratings backward onto earlier matches as though they had already been known.


In [10]:
latest_ratings = (
    pd.Series(final_ratings, name="elo")
    .rename_axis("team")
    .sort_values(ascending=False)
    .reset_index()
)

last_seen = (
    appearances.groupby("team")["date"]
    .max()
    .rename("last_match_date")
)

latest_ratings = latest_ratings.merge(
    last_seen,
    on="team",
    validate="one_to_one",
)

display(latest_ratings.head(30))

print("Feature rows by year:")
display(
    features_df.groupby(features_df["date"].dt.year)
    .size()
    .rename("games")
)

,team,elo,last_match_date
0,Galions,1989.841923,2026-10-22 16:09:25+00:00
1,Gen.G,1974.697651,2026-10-25 22:43:24+00:00
2,Solary,1940.054493,2026-10-22 16:09:25+00:00
3,Bilibili Gaming,1908.462944,2026-10-26 18:03:09+00:00
4,T1 Esports Academy,1893.779310,2026-10-02 12:10:26+00:00
5,Geekay Esports,1874.199156,2025-10-16 19:14:18+00:00
6,G2 Esports,1870.203107,2026-10-25 18:11:29+00:00
7,Dplus Kia,1867.970408,2026-10-25 20:10:09+00:00
8,Hanwha Life Esports,1867.730462,2026-10-25 17:19:41+00:00
9,Movistar KOI Fénix,1858.264806,2026-10-13 17:35:36+00:00


Feature rows by year:


date
2017     5504
2018     6734
2019     8125
2020     9740
2021    12291
2022    12519
2023    11097
2024    10203
2025    10044
2026     9818
Name: games, dtype: int64

## 11. Export features and final state

Save `elo_features_all_years.csv` for modeling and `elo_ratings_all_years.csv` for inspecting final state. Notebook 03 consumes pre-match features. Final ratings describe the end of this dataset, not a historical tournament-start snapshot.


In [11]:
output_dir.mkdir(parents=True, exist_ok=True)

features_path = output_dir / "elo_features_all_years.csv"
ratings_path = output_dir / "elo_ratings_all_years.csv"

features_df.to_csv(features_path, index=False)
latest_ratings.to_csv(ratings_path, index=False)

print("Saved features:", features_path)
print("Saved final ratings:", ratings_path)

Saved features: C:\Users\USER\lol_worlds_predictor\data\processed\elo_features_all_years.csv
Saved final ratings: C:\Users\USER\lol_worlds_predictor\data\processed\elo_ratings_all_years.csv
